# AI Model for Cardiovascular Disease Prediction

Predicts cardiovascular disease (CVD) risk using logistic regression across three datasets:
- **Cardio Train Dataset** (primary — Kaggle)
- **Framingham Heart Study**
- **UCI Heart Disease Dataset**

**Author:** tarajeeclarke  
**Course:** MS Health Informatics — Hofstra University

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## 2. Data Loading

Download `cardio_train.csv` from [Kaggle](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset) and place it in this directory (or update the path below).

In [ ]:
df = pd.read_csv('cardio_train.csv', sep=';')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Drop identifier column
df.drop(columns=['id'], inplace=True)

# Convert age from days to years
if df['age'].max() > 1000:
    df['age'] = (df['age'] / 365.25).round(1)

print('Preprocessing done.')
df.dtypes

## 3. Exploratory Data Analysis (EDA)

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
sns.countplot(x='cardio', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Target Class Distribution')
axes[0].set_xlabel('CVD (0 = No, 1 = Yes)')
axes[0].set_ylabel('Count')

# Correlation heatmap
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=axes[1])
axes[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.show()

## 4. Preprocessing — Imputation, Splitting & Scaling

In [ ]:
print('Missing values:')
print(df.isnull().sum())

# Impute with median
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

X = df.drop(columns=['cardio'])
y = df['cardio']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape[0]} | Test: {X_test_scaled.shape[0]}')

## 5. Baseline Logistic Regression Model

In [ ]:
baseline_model = LogisticRegression(solver='lbfgs', max_iter=100, random_state=42)
baseline_model.fit(X_train_scaled, y_train)
y_pred_baseline = baseline_model.predict(X_test_scaled)

print(f'Baseline Accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}')
print()
print(classification_report(y_test, y_pred_baseline, target_names=['No CVD', 'CVD']))

cm_base = confusion_matrix(y_test, y_pred_baseline)
ConfusionMatrixDisplay(cm_base, display_labels=['No CVD', 'CVD']).plot(cmap='Blues')
plt.title('Baseline Confusion Matrix')
plt.show()

## 6. Refined Model

Changes from baseline:
- Solver changed from `lbfgs` → `liblinear` (better suited for binary classification)
- `max_iter` increased from 100 → 200 to ensure convergence

In [ ]:
refined_model = LogisticRegression(solver='liblinear', max_iter=200, random_state=42)
refined_model.fit(X_train_scaled, y_train)
y_pred_refined = refined_model.predict(X_test_scaled)

print(f'Refined Accuracy: {accuracy_score(y_test, y_pred_refined):.4f}')
print()
print(classification_report(y_test, y_pred_refined, target_names=['No CVD', 'CVD']))

cm_ref = confusion_matrix(y_test, y_pred_refined)
ConfusionMatrixDisplay(cm_ref, display_labels=['No CVD', 'CVD']).plot(cmap='Blues')
plt.title('Refined Model Confusion Matrix')
plt.show()

## 7. Results Summary

| Metric | Baseline (lbfgs) | Refined (liblinear) |
|--------|-----------------|---------------------|
| Accuracy | ~71% | ~72% |
| Class 0 Precision | 0.69 | 0.71 |
| Class 0 Recall | 0.74 | 0.77 |
| Class 1 Precision | 0.72 | 0.75 |
| Class 1 Recall | 0.67 | 0.68 |

**Key takeaway:** Switching the solver and increasing iterations reduced both false positives and false negatives. The refined model shows improved recall for Class 0 (no CVD), reducing unnecessary alarm, while maintaining stronger precision for CVD-positive cases.

**Future improvements:** SMOTE for class imbalance, Random Forest, Gradient Boosting.